In [ ]:
!pip install groq chromadb sentence-transformers pandas -q

print("All packages installed successfully.")
print("Ready for Day 10 - Internship Finale")

All packages installed successfully.
Ready for Day 10 - Internship Finale


In [ ]:
import pandas as pd
import sqlite3
from groq import Groq
import chromadb
import os
import json

print("All imports successfully.")
print("Libraries loaded: pandas, sqlite3, groq, chromadb, os, json")

All imports successfully.
Libraries loaded: pandas, sqlite3, groq, chromadb, os, json


In [ ]:
GROQ_API_KEY = "gsk_EcoTePgPu7Z6xS814LtuWGdyb3FYkclL5nQAUvBpKbxtCsKaQy"
client = Groq(api_key=GROQ_API_KEY)
MODEL =  "llama-3.1-8b-instant"
print("groq client configured.")
print(f"Model: {MODEL}")
print("Status: Ready to generate AI response")

groq client configured.
Model: llama-3.1-8b-instant
Status: Ready to generate AI response


In [ ]:
responsible_system_prompt = """
You are a helpful data analysis assistance.
You only answer questions based on the data provided to you.
If you are not sure about something, say: I do not have enough information to answer that accurately.
Never make up statistics or facts that are not in the data you receive.
"""

test_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "system", "content": responsible_system_prompt},
        {"role": "user", "content": "What is the population of Mars?"}
    ]
)
ai_answer = test_response.choices[0].message.content

print(" === Responsible AI Test ===")
print("Question: What is the population of Mars?")
print("AI Response (with guardrail):")
print(ai_answer)
print("\nNote: A responsible AI should admit it cannot answer this accurately.")

 === Responsible AI Test ===
Question: What is the population of Mars?
AI Response (with guardrail):
I do not have enough information to answer that accurately. You have not provided any data.

Note: A responsible AI should admit it cannot answer this accurately.


In [ ]:
student_df =pd.read_csv("student_performance (3).csv")
print("===Student performance dataset===")
print(f"Shape:{student_df.shape[0]} students, {student_df.shape[1]} columns")
print()
print(student_df.head(5))

===Student performance dataset===
Shape:30 students, 11 columns

   student_id          name  age  gender branch  attendance_pct  \
0           1  Aarav Sharma   20    Male    CSE              85   
1           2   Priya Patel   21  Female    ECE              92   
2           3   Rohit Kumar   20    Male   MECH              67   
3           4    Sneha Iyer   22  Female    CSE              95   
4           5  Vikram Singh   21    Male  CIVIL              72   

   assignment_score  midterm_score  final_score  gpa passed  
0                78             72           76  7.6    Yes  
1                88             85           89  8.9    Yes  
2                55             60           58  5.8    Yes  
3                92             90           94  9.4    Yes  
4                62             65           63  6.3    Yes  


In [ ]:
notes_df = pd.read_csv("college_notes (1).csv")
print(" === College Notes Dataset ===")
print(f"Shape: {notes_df.shape[0]} notes, {notes_df.shape[1]} columns")
print()
print(notes_df[['note_id','subject','topic','difficulty']].to_string(index=False))

 === College Notes Dataset ===
Shape: 15 notes, 6 columns

 note_id             subject                       topic   difficulty
       1     Data Structures                      Arrays     Beginner
       2     Data Structures                Linked Lists     Beginner
       3     Data Structures                Binary Trees Intermediate
       4     Data Structures           Stacks and Queues     Beginner
       5 Database Management                  SQL Basics     Beginner
       6 Database Management               Normalization Intermediate
       7 Database Management                    Indexing Intermediate
       8    Machine Learning                  Regression Intermediate
       9    Machine Learning              Classification Intermediate
      10    Machine Learning                  Clustering     Advanced
      11  Python Programming                   Functions     Beginner
      12  Python Programming Object Oriented Programming Intermediate
      13  Python Programming   

In [ ]:
print("=== data quality report student_performance(1).csv ===")
print("missing values per columns: ")
print(student_df.isnull().sum())
print()

print("data types: ")
print(student_df.dtypes)
print()

print("duplicate rows: ",student_df.duplicated().sum())
print("data quality checks complete")

=== data quality report student_performance(1).csv ===
missing values per columns: 
student_id          0
name                0
age                 0
gender              0
branch              0
attendance_pct      0
assignment_score    0
midterm_score       0
final_score         0
gpa                 0
passed              0
dtype: int64

data types: 
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object

duplicate rows:  0
data quality checks complete


In [ ]:
conn  = sqlite3.connect(':memory:')
student_df.to_sql('students', conn, if_exists='replace', index=False)
print("SQL database created.")
print("Table 'students' loaded with", len(student_df), "rows.")

SQL database created.
Table 'students' loaded with 30 rows.


In [ ]:
query1="""
select branch, count (*) as total_students,
round(avg(gpa),2) as avg_gpa,
round(avg(attendance_pct),1) as avg_attendance
from students
group by branch
order by avg_gpa desc
"""

branch_analysis=pd.read_sql(query1,conn)
print("=== Branch wise Performance analysis====")
print(branch_analysis.to_string(index=False))

=== Branch wise Performance analysis====
branch  total_students  avg_gpa  avg_attendance
    IT               5     8.64            89.4
   CSE              10     7.42            80.0
  MECH               5     7.22            79.4
 CIVIL               4     6.75            75.0
   ECE               6     6.38            69.2


In [ ]:
query2 = """
SELECT name, branch, gpa, attendance_pct, passed
FROM  students
ORDER BY gpa DESC
LIMIT 5
"""

top_students = pd.read_sql(query2, conn)
print("=== Top 5 Students by GPA ===")
print(top_students.to_string(index=False))



=== Top 5 Students by GPA ===
            name branch  gpa  attendance_pct passed
  Meera Krishnan     IT  9.5              96    Yes
      Sneha Iyer    CSE  9.4              95    Yes
Lakshmi Chandran    CSE  9.2              94    Yes
    Swathi Menon     IT  9.1              93    Yes
     Priya Patel    ECE  8.9              92    Yes


In [ ]:
query3="""
select passed, count(*) as student_count,
round(avg(gpa),2) as avg_gpa
from students
group by passed
"""
pass_fail=pd.read_sql(query3, conn)
print("=== Pass/fail statistics===")
print(pass_fail.to_string(index=False))

total = len(student_df)
passes = len(student_df[student_df['passed'] == 'yes'])
pass_rate = round((passes/total)* 100, 1)
print(f"\nOverall Pass Rate: {pass_rate}%({passes}/{total})")

=== Pass/fail statistics===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61

Overall Pass Rate: 0.0%(0/30)


In [ ]:
branch_summary = ""
for _, row in branch_analysis.iterrows():
  branch_summary += f" - {row['branch']}: {row['total_students']} students, Avg GPA: {row['avg_gpa']}, Avg Attendance: {row['avg_attendance']}%\n"

top_summary = ""
for _, row in top_students.iterrows():
  top_summary += f" - {row['name']} ({row['branch']}): GPA {row['gpa']}%\n"

data_summary = f"""
STUDENT PERFORMANCE DATA SUMMARY
Total Students: {total}
Overall Pass Rate: {pass_rate}%
Average GPA actoss all students: {round(student_df['gpa'].mean(), 2)}

Performance by Branch:
{branch_summary}
Top 5 Students by GPA:
{top_summary}
"""

print("=== Data Summary ===")
print(data_summary)


=== Data Summary ===

STUDENT PERFORMANCE DATA SUMMARY
Total Students: 30
Overall Pass Rate: 0.0%
Average GPA actoss all students: 7.29

Performance by Branch:
 - IT: 5 students, Avg GPA: 8.64, Avg Attendance: 89.4%
 - CSE: 10 students, Avg GPA: 7.42, Avg Attendance: 80.0%
 - MECH: 5 students, Avg GPA: 7.22, Avg Attendance: 79.4%
 - CIVIL: 4 students, Avg GPA: 6.75, Avg Attendance: 75.0%
 - ECE: 6 students, Avg GPA: 6.38, Avg Attendance: 69.2%

Top 5 Students by GPA:
 - Meera Krishnan (IT): GPA 9.5%
 - Sneha Iyer (CSE): GPA 9.4%
 - Lakshmi Chandran (CSE): GPA 9.2%
 - Swathi Menon (IT): GPA 9.1%
 - Priya Patel (ECE): GPA 8.9%




In [ ]:
user_message = f"""
Here is the student performance data for this semester:
{data_summary}
Please provide:
1. Three key insights from this data
2. Which branch needs the mose improvement?
3. One recommendation for the college principal
"""
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "system", "content": responsible_system_prompt},
        {"role": "user", "content": user_message}
    ]
)
ai_analysis = response.choices[0].message.content
print("="*60)
print("=== AI Analysis ===")
print("="*60)
print(ai_analysis)

=== AI Analysis ===
Based on the provided data, here are the requested insights and recommendation:

1. **Three key insights:**

- The overall pass rate is 0.0%, indicating a high failure rate, which is a significant concern.
- There is a significant variation in average GPA and attendance across different branches. The IT branch has the highest average GPA, while the ECE branch has the lowest.
- Top-performing students are concentrated in the IT and CSE branches, suggesting that these branches have a more favorable learning environment or that these students are more motivated.

2. **Branch that needs the most improvement:**

Based on the data, **ECE (Electrical and Computer Engineering) branch** needs the most improvement. It has the lowest average GPA (6.38) and the lowest attendance rate (69.2%) among all branches.

3. **Recommendation for the college principal:**

Based on the data, I would recommend that the college principal consider implementing targeted support and resources f